# Stellar Coordinate Explorer - Data Loading & Inspection

## Objective 
Load a subset of Gaia star catalog data and perform initial inspection.

## Dataset
- Source: Gaia DR3
- Subset: ~ 10,000 stars
- Filters:
  - $Magnitude\ \lt 10$
  - $Parallax\ \gt5\ mas\ (\approx 200\ parsecs)$

The Dataset was obtained from the [Gaia archive](https://gea.esac.esa.int/archive/) using the following SQL query:
```SQL
SELECT TOP 10000
    source_id,
    ra, 
    dec, 
    parallax,
    phot_g_mean_mag,
    bp_rp
FROM gaiadr3.gaia_source
WHERE phot_g_mean_mag < 10
AND parallax > 5
```

## Why this matters for radio astonomy
- Before observing, you load a catalog of potential calibrators or targets
- Inspecting columns reveals units (e.g., ra/dec in degrees, parallax in mas)
- Filtering on magnitude selects sources strong enough to detect

## Goals for Today
- Load dataset
- Inspect structure
- Understand key columns
- Print first 5 rows
- Create a filtered table with values below a magnitude
- Print the 5 brightest stars with their magnitudes
- Print the mean and median of the full sample
- Save the filtered table for Day 4

In [1]:
# Imports
from astropy.table import Table
import numpy as np
import pandas as pd

In [2]:
# Load the FITS table
table = Table.read('../../data/gaia_subset.fits')


# Get summary information
print(f"{'#' * 20} Summary information {'#' * 20}")
table.info()

#################### Summary information ####################
<Table length=10000>
      name       dtype  unit    class     n_bad
--------------- ------- ---- ------------ -----
      source_id   int64            Column     0
             ra float64  deg       Column     0
            dec float64  deg       Column     0
       parallax float64  mas       Column     0
phot_g_mean_mag float32  mag       Column     0
          bp_rp float32  mag MaskedColumn    14


There appears to be 14 missing values in the `bp_rp` column. `bp_rp` is difference between the integrated magintude of the source measured in the blue photometer and red photometer.

### Key Columns

Looking at the columns closely:

In [3]:
print(f"\n{'#' * 20} Show table column names {'#' * 20}")
print(table.columns) 
print(table.colnames)


#################### Show table column names ####################
<TableColumns names=('source_id','ra','dec','parallax','phot_g_mean_mag','bp_rp')>
['source_id', 'ra', 'dec', 'parallax', 'phot_g_mean_mag', 'bp_rp']


We have the `source_id` used to identify the star, `ra` and `dec` which is the right ascension and declination coordinates of the source (using the ICRS coordinate frame), `parallax` (for determining distance to source), `phot_g_mean_mag` (mean G-band magnitude) and `bp_rp`. Therefore key columns can be summarized as:

- `ra`, `dec`: Equitorial coordinates (degrees)
- `parallax`: Used to estimate distance
- `phot_g_mean_mag`: Used to determine brightness
- `bp_rp`: Color index useful for indicating temperature.


Now, looking at the first 5 rows:

In [4]:
print(f"{'#' * 20} First five rows {'#' * 20}")
table[:5]

#################### First five rows ####################


source_id,ra,dec,parallax,phot_g_mean_mag,bp_rp
,deg,deg,mas,mag,mag
int64,float64,float64,float64,float32,float32
16733192740608,45.152965337754885,0.3863421150872851,5.473905596643335,9.925234,0.7235527
407785670617600,44.263446403533244,1.5015924603581685,5.327594570736073,9.749034,0.5830326
2517233088211712,48.307934519908706,3.82011414311565,16.796145185469527,9.589524,1.2472496
2710712774989440,48.1134802323883,3.964476089897251,8.89349519082703,9.847445,0.8104763
3504422731187584,46.25523412225751,4.465545045836133,5.850867569879387,9.01819,1.1034575


We see that the `ra` and `dec` are in degrees, `parallax` is in milliarcsecs (mas) and both `phot_g_mean_mag` and `bp_rp` are magnitudes. 

Let's filter to obtain the stars with magnitudes `phot_g_mean_mag` $\lt 5$

In [5]:
mask = table["phot_g_mean_mag"] < 5
print(f"{'#' * 20} Stars with G-band mean magnitudes less than 5 {'#' * 20}")
bright_table = table[mask]
bright_table

#################### Stars with G-band mean magnitudes less than 5 ####################


source_id,ra,dec,parallax,phot_g_mean_mag,bp_rp
,deg,deg,mas,mag,mag
int64,float64,float64,float64,float32,float32
65283232316451328,56.45679586432521,24.36754142471917,7.671309961502887,3.863349,-0.013768196
66526127137440128,57.29068893293528,24.053211829998986,8.118448445799721,3.6157947,-0.036376238
128855517166594048,41.978024427837525,29.24656343803326,19.038950218744798,4.197216,1.2571971
11166438229308672,51.20302758439933,9.028538893521931,15.078813297551084,3.3823738,1.1385627
65271996684817280,56.2190045645284,24.11313304360812,8.345685150896063,3.6982086,-0.08796215
140457735662620800,42.87854040758558,35.05946084085705,7.776155027856757,3.901517,1.8479674
12730974556132096,51.7925714099467,9.732550147153876,16.791442221122395,3.7018294,-0.05780959
21859669845327488,41.236830226139006,10.113961719979088,37.5923954771885,4.1537447,0.44148636


Let's print the 5 brightest stars

In [6]:
sorted_table = bright_table.copy()
sorted_table.sort('phot_g_mean_mag')
print(f"{'#' * 20} Five brightest stars {'#' * 20}")
sorted_table[:5]

#################### Five brightest stars ####################


source_id,ra,dec,parallax,phot_g_mean_mag,bp_rp
,deg,deg,mas,mag,mag
int64,float64,float64,float64,float32,float32
418551920284673408,10.12724197930297,56.53718879378639,14.090976249252234,1.9425238,1.1434835
160886283751041408,74.24843702624405,33.16601472318802,7.249065034494049,2.1975422,1.4451852
447071293401293056,46.19914439438418,53.50642323794812,14.125159961030231,2.651,1.0181456
779106556394253824,167.41547770774767,44.4983642995888,23.227226550456237,2.68003,1.3199959
510204838759030144,21.45661245731054,60.235067611380984,32.02972419177956,2.6805298,0.5723715


In [ ]:
## Save the brightest stars
sorted_table.write("../../data/bright_stars_filtered.fits")

Now we obtain the mean and median G-band magnitudes of the whole set:

In [8]:
mean_mag = np.mean(table['phot_g_mean_mag'])
median_mag = np.median(table['phot_g_mean_mag'])

print(f"{'#' * 20} Mean and Median G-band mean magnitudes of the 10000 sources {'#' * 20}")
print(f"Mean magnitude: {mean_mag}")
print(f"Median magnitude: {median_mag}")

#################### Mean and Median G-band mean magnitudes of the 10000 sources ####################
Mean magnitude: 8.629207611083984
Median magnitude: 8.967992782592773
